# Factor individual de implicación (`w`)
**Coevaluación 360 · Modelos Lineales Generalizados**

Lee la hoja de respuestas del formulario y devuelve, para cada estudiante y proyecto,
el factor `w ∈ [0, 1]` que pondera la nota del equipo.

El cálculo tiene tres pasos: se normaliza lo que **emite** cada evaluador (así su
severidad o generosidad deja de influir), se toma la **mediana** de lo que **recibe**
cada estudiante, y se convierte en factor con `w = min(1, R / 0,80)`.

## 1 · Núcleo del cálculo

In [ ]:
import re, unicodedata
import numpy as np, pandas as pd

P1W, P2W = 2, 1        # 'su parte' cuenta doble; 'la parte de los demás', simple
TOL      = 0.80        # franja de indiferencia
TOPE     = 0.80        # tope de w para quien no cumplimenta
BLOQUES  = ["A", "C1", "C2", "C3"]   # A = autoevaluación

def nivel(x):
    """'2. En gran parte' -> 2.0"""
    m = re.match(r"\s*(\d)", str(x))
    return float(m.group(1)) if m else np.nan

def normalizar(x):
    """Los nombres viajan como texto entre el bloque A y los bloques C:
    sin normalizar, el mismo estudiante genera dos identidades distintas."""
    x = unicodedata.normalize("NFKD", str(x)).encode("ascii", "ignore").decode()
    return " ".join(x.upper().split())

def calcular(resp):
    # 1) si alguien reenvía el formulario, vale su último envío
    resp = (resp.sort_values("Marca temporal")
                .drop_duplicates(subset=["A. NOMBRE", "PROYECTO"], keep="last"))

    # 2) de formato ancho a un registro por par (evaluador, evaluado)
    filas = []
    for _, r in resp.iterrows():
        evaluador = normalizar(r["A. NOMBRE"])
        for b in BLOQUES:
            nombre = r.get(f"{b}. NOMBRE", "")
            p1, p2 = nivel(r.get(f"{b}. CUMPLIMIENTO")), nivel(r.get(f"{b}. COLABORACIÓN"))
            if str(nombre).strip() == "" or np.isnan(p1) or np.isnan(p2):
                continue
            filas.append(dict(
                proyecto=r["PROYECTO"], equipo=r["EQUIPO"],
                evaluador=evaluador, evaluado=normalizar(nombre),
                p1=p1, p2=p2, s=(P1W * p1 + P2W * p2) / (P1W + P2W),
                matiz=r.get(f"{b}. MATIZ", "")))
    L = pd.DataFrame(filas)

    # 3) normalización por evaluador (algoritmo WebPA): cada quien reparte, no califica.
    #    m es cuánta gente ha valorado ESE evaluador, no el tamaño del equipo, de modo
    #    que sus índices promedian 1 aunque haya dejado a alguien sin valorar.
    g = L.groupby(["proyecto", "equipo", "evaluador"])["s"]
    L["m"], L["suma"] = g.transform("size"), g.transform("sum")
    L["omega"] = np.where(L["suma"] == 0, 1.0, L["s"] / L["suma"] * L["m"])

    # 4) agregación por evaluado y factor
    F = (L.groupby(["proyecto", "equipo", "evaluado"])
           .agg(n_votos=("omega", "size"),
                R=("omega", "median"),
                discrepancia=("omega", lambda v: v.max() - v.min()))
           .reset_index())
    F["w"] = np.minimum(1, F["R"] / TOL)

    # 5) salvaguardas
    emisores = set(map(tuple, L[["proyecto", "equipo", "evaluador"]].drop_duplicates().values))
    F["respondio"] = [tuple(x) in emisores for x in F[["proyecto", "equipo", "evaluado"]].values]
    F.loc[~F["respondio"], "w"] = np.minimum(F.loc[~F["respondio"], "w"], TOPE)
    F["aviso"] = np.select(
        [~F["respondio"], F["n_votos"] < 3, F["discrepancia"] > 0.60, F["w"] < 1],
        ["no cumplimentó la coevaluación",
         "menos de tres votos: la mediana no protege",
         "evaluadores muy discrepantes",
         "descuento pendiente de confirmar con bitácora"], default="")
    F = F.round({"R": 3, "discrepancia": 3, "w": 2})
    return F.sort_values(["proyecto", "equipo", "w"]), L

## 2 · Comprobación con casos conocidos

Antes de tocar datos reales: cinco escenarios cuyo resultado está calculado a mano.
Si esta celda no imprime `TODO CORRECTO`, no sigas.

In [ ]:
pleno, medias, nada = (3, 3), (1, 1), (0, 0)

def fila(equipo, quien, votos):
    """Construye una respuesta en el formato ancho del formulario."""
    r = {"Marca temporal": "2026-01-01", "PROYECTO": "P1", "EQUIPO": equipo}
    orden = [quien] + [k for k in votos if k != quien]
    for b, nombre in zip(BLOQUES, orden):
        p1, p2 = votos[nombre]
        r[f"{b}. NOMBRE"] = nombre
        r[f"{b}. CUMPLIMIENTO"] = f"{p1}. x"
        r[f"{b}. COLABORACIÓN"] = f"{p2}. x"
    for b in BLOQUES[len(orden):]:
        r[f"{b}. NOMBRE"] = ""
        r[f"{b}. CUMPLIMIENTO"] = ""
        r[f"{b}. COLABORACIÓN"] = ""
    return r

pruebas = {
    "equipo equilibrado":
        ([fila("E1", q, {"A": pleno, "B": pleno, "C": pleno}) for q in "ABC"],
         {"A": 1.00, "B": 1.00, "C": 1.00}),
    "C free-rider y se autoinfla":
        ([fila("E2", "A", {"A": pleno, "B": pleno, "C": medias}),
          fila("E2", "B", {"A": pleno, "B": pleno, "C": medias}),
          fila("E2", "C", {"A": medias, "B": medias, "C": pleno})],
         {"A": 1.00, "B": 1.00, "C": 0.54}),
    "voto hostil aislado de A contra C":
        ([fila("E3", "A", {"A": pleno, "B": pleno, "C": nada}),
          fila("E3", "B", {"A": pleno, "B": pleno, "C": pleno}),
          fila("E3", "C", {"A": pleno, "B": pleno, "C": pleno})],
         {"A": 1.00, "B": 1.00, "C": 1.00}),
    "equipo de cuatro, D a medias":
        ([fila("E4", q, {"A": pleno, "B": pleno, "C": pleno, "D": medias}) for q in "ABCD"],
         {"A": 1.00, "B": 1.00, "C": 1.00, "D": 0.50}),
    "C no cumplimenta el formulario":
        ([fila("E5", "A", {"A": pleno, "B": pleno, "C": pleno}),
          fila("E5", "B", {"A": pleno, "B": pleno, "C": pleno})],
         {"A": 1.00, "B": 1.00, "C": 0.80}),
}

fallos = 0
for nombre, (filas, esperado) in pruebas.items():
    obtenido = calcular(pd.DataFrame(filas))[0].set_index("evaluado")["w"].to_dict()
    ok = all(abs(obtenido[k] - v) < 0.005 for k, v in esperado.items())
    fallos += not ok
    print(f"{'ok  ' if ok else 'FALLO'} {nombre:<36} {obtenido}")

print("\nTODO CORRECTO" if fallos == 0 else f"\n{fallos} PRUEBAS FALLIDAS")

## 3 · Lectura de la hoja de respuestas

Autentica con la cuenta propietaria de la hoja y localiza la pestaña de respuestas
por su cabecera (`Marca temporal`), no por su nombre, que Forms cambia según el idioma.

In [ ]:
from google.colab import auth
from google.auth import default
import gspread

HOJA = "1amIYvjvIkTcYjbrcv1traf9ynIkIBpBLzErMR5s6oY4"

auth.authenticate_user()
gc = gspread.authorize(default()[0])

pestanas = gc.open_by_key(HOJA).worksheets()
hoja = next(p for p in pestanas if "Marca temporal" in p.row_values(1))
valores = hoja.get_all_values()
respuestas = pd.DataFrame(valores[1:], columns=valores[0])

print(f"Pestaña '{hoja.title}': {len(respuestas)} respuestas")
respuestas.head()

> **Si prefieres no autenticar**, descarga la pestaña de respuestas como CSV
> (*Archivo › Descargar › .csv*), súbela a Colab y sustituye la celda anterior por
> `respuestas = pd.read_csv("coevaluacion_respuestas.csv")`.

## 4 · Cálculo y revisión

In [ ]:
factores, largo = calcular(respuestas)
factores

Ningún `w < 1` se aplica solo con los votos. Estas son las líneas que el formulario
exige cuando alguien marca 1 o 0, y son el material con el que se resuelve la revisión
contra la bitácora:

In [ ]:
justificaciones = largo.query("p1 <= 1 or p2 <= 1")[
    ["proyecto", "equipo", "evaluado", "evaluador", "p1", "p2", "matiz"]]
justificaciones

## 5 · Exportar

In [ ]:
factores.to_csv("coevaluacion_factores.csv", index=False)

from google.colab import files
files.download("coevaluacion_factores.csv")